# Guia TNF: asistente turistico de Tenerife

Este notebook sera el entregable principal de la practica. El objetivo es construir un asistente guia turistico de Tenerife que combine recuperacion de informacion sobre el PDF proporcionado por el profesor, dialogo multiturno con mantenimiento de contexto, streaming y al menos una tool o llamada externa.


## 0. Preparacion del entorno

El notebook espera una variable `OPENAI_API_KEY`. Puedes cargarla desde un archivo `.env` situado junto al notebook. Los modelos se configuran mediante variables de entorno para facilitar cambios sin tocar todas las celdas. Por defecto usaremos un modelo pequeno durante el desarrollo y comparar ya que el numero de token proporcionado es limitado.


In [ ]:
# Si estas en un entorno limpio, descomenta esta celda.
# Despues de instalar, puede ser necesario reiniciar el kernel.
# %pip install -q -U openai openai-agents mcp python-dotenv pandas pydantic graphviz tiktoken ipykernel


### 0.1. Imports y configuracion comun

In [ ]:
import getpass
import json
import os
import time
from pathlib import Path
from typing import Any, Literal

import pandas as pd
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field


In [ ]:
def find_project_dir() -> Path:
    current = Path.cwd()
    if (current / "guia_tnf.ipynb").exists():
        return current
    return current


PROJECT_DIR = find_project_dir()
load_dotenv(PROJECT_DIR / ".env")

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-4.1-mini")
FAST_MODEL = os.getenv(
    "OPENAI_FAST_MODEL",
    os.getenv("OPENAI_EVALUATION_MODEL", GENERATION_MODEL),
)
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

client = OpenAI()
GUIDE_PATH = PROJECT_DIR / "TENERIFE.pdf"

print("Directorio del proyecto:", PROJECT_DIR)
print("Guia turistica:", GUIDE_PATH)
print("Modelo generativo:", GENERATION_MODEL)
print("Modelo rapido:", FAST_MODEL)
print("Modelo de embeddings:", EMBEDDING_MODEL)

if not GUIDE_PATH.exists():
    print("Aviso: no se encontro la guia turistica esperada en", GUIDE_PATH)
